# HandVid — AI Product Hands
**Adds natural AI-generated hands cradling your product into any video.**

### Quick Start
1. `Runtime → Change runtime type → T4 GPU` (free)
2. Run all cells top to bottom (`Runtime → Run all`)
3. Upload your product video when prompted
4. Download the result from the last cell

---
> **First run:** installs packages + downloads ~6 GB of models (~8 min on Colab).  
> **Subsequent runs in same session:** model loading only (~30 sec).

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    name, vram, cc = [x.strip() for x in result.stdout.strip().split(',')]
    print(f'✓ GPU : {name}')
    print(f'  VRAM: {vram}')
    print(f'  CUDA compute: {cc}')
else:
    print('✗ No GPU detected!')
    print('  Go to: Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# No version pins — let pip resolve against Colab's pre-installed packages
import subprocess, sys

def pip(*pkgs, label=None):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + list(pkgs)
    r = subprocess.run(cmd, capture_output=True, text=True)
    lbl = label or pkgs[0].split('/')[-1]
    icon = '✓' if r.returncode == 0 else '✗ FAILED'
    print(f'  {icon}  {lbl}')
    if r.returncode != 0:
        print('       ', r.stderr[-400:])

print('Upgrading packages to latest compatible versions...')
pip('huggingface_hub')
pip('diffusers')
pip('transformers')
pip('accelerate')
pip('peft')
pip('git+https://github.com/facebookresearch/segment-anything.git', label='segment-anything')
pip('opencv-python-headless')
pip('ffmpeg-python')
pip('scipy')

print()
print('=' * 50)
print('IMPORTANT: Restart the runtime now!')
print('  Runtime → Restart session')
print('  Then run Cell 3 (skip Cell 2).')
print('=' * 50)


In [ ]:
# ── Cell 3: Load models ───────────────────────────────────────────────────────
# Uses only freely available models — no HuggingFace login required
# First run: ~6 GB download (~8 min). Cached for the rest of the session.

import os, urllib.request, torch
from pathlib import Path
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetInpaintPipeline,
    UniPCMultistepScheduler,
)

MODELS_DIR = Path('/content/models')
MODELS_DIR.mkdir(exist_ok=True)

# ── SAM vit_b (375 MB, no auth needed) ───────────────────────────────────────
SAM_PATH = MODELS_DIR / 'sam_vit_b_01ec64.pth'
if not SAM_PATH.exists():
    print('Downloading SAM vit_b (375 MB)...')
    def _hook(b, bs, total):
        pct = min(b*bs/total*100, 100)
        print(f'\r  {pct:.0f}%', end='', flush=True)
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        SAM_PATH, reporthook=_hook
    )
    print('\n✓ SAM downloaded')
else:
    print('✓ SAM already cached')

# ── ControlNet openpose for SD 1.5 (no auth needed) ──────────────────────────
dtype = torch.float16
print('\nLoading ControlNet openpose (~1.7 GB)...')
controlnet = ControlNetModel.from_pretrained(
    'lllyasviel/control_v11p_sd15_openpose',
    torch_dtype=dtype,
)

# ── SD 1.5 inpainting — Dreamshaper 8 (no auth, freely available) ────────────
# Dreamshaper produces more photorealistic hands than the base SD model.
print('Loading Dreamshaper-8 inpainting (~2.1 GB)...')
pipe = StableDiffusionControlNetInpaintPipeline.from_pretrained(
    'Lykon/dreamshaper-8-inpainting',
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None,
    requires_safety_checker=False,
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to('cuda')
pipe.enable_attention_slicing()   # saves VRAM without needing xformers

# ── Load SAM predictor ────────────────────────────────────────────────────────
print('Loading SAM predictor...')
from segment_anything import sam_model_registry, SamPredictor
sam = sam_model_registry['vit_b'](checkpoint=str(SAM_PATH))
sam.to('cuda')
sam_predictor = SamPredictor(sam)

print('\n✓ All models loaded and ready!')


In [ ]:
# ── Cell 4: Pipeline functions ────────────────────────────────────────────────
import cv2, os
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter

# ─── Segmentation ────────────────────────────────────────────────────────────
def get_product_bbox(frame_bgr, point=None):
    h, w = frame_bgr.shape[:2]
    if point is None:
        point = (w // 2, h // 2)
    try:
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        sam_predictor.set_image(frame_rgb)
        masks, scores, _ = sam_predictor.predict(
            point_coords=np.array([[point[0], point[1]]]),
            point_labels=np.array([1]),
            multimask_output=True,
        )
        mask = (masks[np.argmax(scores)] * 255).astype(np.uint8)
        coords = cv2.findNonZero(mask)
        x, y, mw, mh = cv2.boundingRect(coords)
        # Add a small margin
        pad = 10
        return (max(0,x-pad), max(0,y-pad), min(w,x+mw+pad), min(h,y+mh+pad))
    except Exception as e:
        print(f'  SAM fallback ({e})')
        m = 0.15
        return (int(w*m), int(h*m), int(w*(1-m)), int(h*(1-m)))


# ─── Hand pose ───────────────────────────────────────────────────────────────
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17),
]
FINGER_COLORS = [(255,0,0),(255,128,0),(255,255,0),(0,255,0),(0,128,255)]

def make_cradle_keypoints(wx, wy, scale, is_right, curl=0.4):
    """
    Palm-up cradling pose.
    Wrist anchor (wx, wy) is at the bottom corner of the product.
    Fingers point UPWARD (negative dy) to cup around the product bottom.
    """
    s = scale
    f = 1 if is_right else -1
    # dy negative = upward in image coords
    raw = [
        (0.0,  0.0),                                             # 0 wrist
        ( f*0.30, -0.15),( f*0.55,-0.30),( f*0.70,-0.42),( f*0.80,-0.54),  # thumb 1-4
        ( f*0.15, -0.50),( f*0.17,-0.75),( f*0.18,-0.95+curl*0.25),( f*0.18,-1.10+curl*0.50),  # index 5-8
        ( f*0.02, -0.55),( f*0.02,-0.82),( f*0.02,-1.03+curl*0.25),( f*0.02,-1.20+curl*0.50),  # middle 9-12
        (-f*0.13, -0.52),(-f*0.14,-0.78),(-f*0.14,-0.98+curl*0.25),(-f*0.14,-1.14+curl*0.50), # ring 13-16
        (-f*0.26, -0.45),(-f*0.28,-0.65),(-f*0.30,-0.83+curl*0.25),(-f*0.30,-0.97+curl*0.50), # pinky 17-20
    ]
    return np.array([[wx + dx*s, wy + dy*s] for dx,dy in raw], dtype=np.float32)

def draw_hand(canvas, pts, thickness=3):
    H, W = canvas.shape[:2]
    ranges = [(1,4),(5,8),(9,12),(13,16),(17,20)]
    def cl(p):
        return (int(np.clip(p[0],0,W-1)), int(np.clip(p[1],0,H-1)))
    for i,j in HAND_CONNECTIONS:
        color = (255,255,255)
        for fi,(s,e) in enumerate(ranges):
            if s<=i<=e or s<=j<=e:
                color = FINGER_COLORS[fi]; break
        cv2.line(canvas, cl(pts[i]), cl(pts[j]), color, thickness, cv2.LINE_AA)
    for pt in pts:
        cv2.circle(canvas, cl(pt), thickness+1, (255,255,255), -1, cv2.LINE_AA)

def generate_pose_and_mask(image_shape, bbox, hand_scale=0.18, curl=0.4):
    H, W = image_shape[:2]
    x1, y1, x2, y2 = bbox
    prod_w = x2 - x1
    prod_h = y2 - y1

    # Hand scale based on product width
    s = max(prod_w * hand_scale * 4, 40)   # min 40px so hands are always visible

    # Wrists sit at the bottom corners of the product, slightly inward
    # This keeps hands INSIDE the frame at all times
    wr_x = x2 - prod_w * 0.05   # right wrist near bottom-right of product
    wl_x = x1 + prod_w * 0.05   # left wrist near bottom-left of product
    w_y  = min(y2 + s * 0.10, H - 5)   # just below product bottom, clamped to frame

    pts_r = make_cradle_keypoints(wr_x, w_y, s, is_right=True,  curl=curl)
    pts_l = make_cradle_keypoints(wl_x, w_y, s, is_right=False, curl=curl)

    # Clamp all keypoints to frame
    for pts in (pts_r, pts_l):
        pts[:,0] = np.clip(pts[:,0], 0, W-1)
        pts[:,1] = np.clip(pts[:,1], 0, H-1)

    # Pose skeleton image
    pose = np.zeros((H, W, 3), dtype=np.uint8)
    draw_hand(pose, pts_r)
    draw_hand(pose, pts_l)

    # Inpaint mask — region where hands appear
    mask = np.zeros((H, W), dtype=np.uint8)
    all_pts = np.vstack([pts_r, pts_l]).astype(np.int32)
    hull = cv2.convexHull(all_pts)
    cv2.fillConvexPoly(mask, hull, 255)

    # Extend mask to cover the bottom portion of the product
    # (fingers overlap the bottom edge of product — that's what "cradling" looks like)
    overlap_top = max(y1, y2 - int(prod_h * 0.30))  # bottom 30% of product
    mask[overlap_top:H, max(0,x1-15):min(W,x2+15)] = 255

    # Dilate for smooth edges
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (20, 20))
    mask = cv2.dilate(mask, kernel)

    # Verify mask is not empty
    if mask.sum() == 0:
        print('  Warning: mask is empty — using fallback strip at product bottom')
        mask[max(0,y2-40):min(H,y2+60), max(0,x1-20):min(W,x2+20)] = 255

    return pose, mask, pts_r, pts_l


# ─── Hand generation ──────────────────────────────────────────────────────────
SKIN_DESCS = {
    'light':  'fair skin, light complexion',
    'medium': 'medium skin tone, natural complexion',
    'dark':   'dark skin, deep complexion',
}

def generate_hands(frame_rgb, mask, pose_img, skin_tone='medium', steps=25, seed=42):
    if mask.sum() == 0:
        print('  Skipping frame — mask is empty')
        return frame_rgb

    skin = SKIN_DESCS.get(skin_tone, 'natural skin tone')
    prompt = (
        f'photorealistic human hands gently cradling and holding a product from below, '
        f'palms up, fingers curled, {skin}, well-manicured, studio lighting, '
        f'sharp focus, 8k, high detail, realistic skin texture'
    )
    neg = (
        'extra fingers, missing fingers, deformed hands, mutated, bad anatomy, '
        'cartoon, anime, blurry, low quality, watermark, gloves, robot, '
        'disconnected limbs, floating hands'
    )
    h, w = frame_rgb.shape[:2]
    scale = min(768 / max(h, w), 1.0)
    tw = max((int(w*scale)//8)*8, 512)
    th = max((int(h*scale)//8)*8, 512)

    pil_img  = Image.fromarray(frame_rgb).resize((tw,th), Image.LANCZOS)
    pil_mask = Image.fromarray(mask).resize((tw,th), Image.NEAREST)
    pil_pose = Image.fromarray(pose_img).resize((tw,th), Image.LANCZOS)

    gen = torch.Generator('cuda').manual_seed(seed)
    with torch.autocast('cuda'):
        result = pipe(
            prompt=prompt, negative_prompt=neg,
            image=pil_img, mask_image=pil_mask, control_image=pil_pose,
            num_inference_steps=steps, guidance_scale=7.5,
            controlnet_conditioning_scale=0.85, generator=gen,
        ).images[0]

    result = result.resize((w, h), Image.LANCZOS)
    result_np = np.array(result)

    # Soft blend only in the masked region
    mask_soft = gaussian_filter(mask.astype(np.float32), sigma=4) / 255.0
    mask_soft = np.clip(mask_soft, 0, 1)
    m3 = np.stack([mask_soft]*3, axis=-1)
    composite = (result_np * m3 + frame_rgb * (1 - m3)).astype(np.uint8)
    return composite


# ─── Video I/O ────────────────────────────────────────────────────────────────
def extract_frames(path):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frames = []
    while True:
        ret, f = cap.read()
        if not ret: break
        frames.append(f)
    cap.release()
    return frames, fps

def save_video(frames_bgr, path, fps):
    h, w = frames_bgr[0].shape[:2]
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w,h))
    for f in frames_bgr:
        writer.write(f)
    writer.release()
    tmp = path.replace('.mp4','_tmp.mp4')
    os.rename(path, tmp)
    os.system(f'ffmpeg -y -i {tmp} -c:v libx264 -pix_fmt yuv420p -crf 18 {path} -loglevel quiet')
    os.remove(tmp)

print('✓ Pipeline functions loaded')


In [ ]:
# ── Cell 5: Upload your product video ────────────────────────────────────────
from google.colab import files
import io

print('Click "Choose Files" and select your product video (MP4, MOV, AVI)...')
uploaded = files.upload()

if uploaded:
    VIDEO_NAME = list(uploaded.keys())[0]
    VIDEO_PATH = f'/content/{VIDEO_NAME}'
    with open(VIDEO_PATH, 'wb') as f:
        f.write(uploaded[VIDEO_NAME])
    print(f'\n✓ Uploaded: {VIDEO_NAME}')
    
    # Show first frame as preview
    cap = cv2.VideoCapture(VIDEO_PATH)
    ret, frame = cap.read()
    cap.release()
    if ret:
        from IPython.display import display
        display(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        print(f'  Resolution: {frame.shape[1]}×{frame.shape[0]}')
else:
    print('No file uploaded.')

In [ ]:
# ── Cell 6: Settings — adjust before running ──────────────────────────────────

# ┌─────────────────────────────────────────────────────┐
# │  Tune these to your product                         │
# └─────────────────────────────────────────────────────┘

SKIN_TONE      = 'medium'   # 'light' | 'medium' | 'dark'
HAND_SIZE      = 0.18       # 0.10 (small) → 0.30 (large), relative to product width
FINGER_CURL    = 0.35       # 0.1 (flat palm) → 0.7 (cupped hands)
INFERENCE_STEPS = 25        # 20=fast/ok · 30=balanced · 40=best quality (slower)
SEED           = 42         # fixed seed keeps hands consistent across frames

# Optional: if product detection is off, click on the product
# in the preview above and set SAM_POINT = (x, y)
SAM_POINT = None            # e.g. (320, 240) or leave None for auto (image center)

print('Settings:')
print(f'  Skin tone      : {SKIN_TONE}')
print(f'  Hand size      : {HAND_SIZE}')
print(f'  Finger curl    : {FINGER_CURL}')
print(f'  Inference steps: {INFERENCE_STEPS}')
print(f'  Seed           : {SEED}')

In [ ]:
# ── Cell 7: Preview pose on first frame ──────────────────────────────────────
from IPython.display import display

cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
cap.release()

frame_rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
h, w = frame_rgb.shape[:2]

bbox = get_product_bbox(first_frame, point=SAM_POINT)
x1,y1,x2,y2 = bbox
print(f'Image size  : {w} × {h}')
print(f'Product bbox: ({x1},{y1}) → ({x2},{y2})  size: {x2-x1}×{y2-y1}')

pose_img, mask, pts_r, pts_l = generate_pose_and_mask(
    (h, w), bbox, hand_scale=HAND_SIZE, curl=FINGER_CURL
)

mask_px = int(mask.sum() / 255)
print(f'Mask pixels : {mask_px}  (should be > 1000 — if 0 hands will not appear)')

if mask_px == 0:
    print('  ✗ Mask is EMPTY — adjust SAM_POINT or HAND_SIZE in Cell 6')
else:
    print('  ✓ Mask looks good')

# Visualise
vis = frame_rgb.copy()
tint = np.zeros_like(vis)
tint[mask > 128] = [0, 100, 255]
vis = cv2.addWeighted(vis, 0.65, tint, 0.35, 0)
vis = cv2.addWeighted(vis, 1.0, pose_img, 0.9, 0)
cv2.rectangle(vis, (x1,y1), (x2,y2), (0,255,0), 3)

# Show wrist positions
wr = pts_r[0].astype(int)
wl = pts_l[0].astype(int)
cv2.circle(vis, tuple(wr), 8, (255,0,255), -1)
cv2.circle(vis, tuple(wl), 8, (255,0,255), -1)

print()
print('Green box = product  |  Blue tint = inpaint region  |  Magenta dots = wrists')
display(Image.fromarray(vis).resize((min(w,900), int(h*min(900/w,1))), Image.LANCZOS))


In [ ]:
# ── Cell 8: Run full pipeline ─────────────────────────────────────────────────
# Processes every frame. Time: ~15-30 sec/frame on T4.
# For a 30-frame clip → ~8-15 min.
# Tip: trim your video to 5-10 sec for faster results.

from tqdm.notebook import tqdm

OUTPUT_PATH = '/content/handvid_output.mp4'

print(f'Extracting frames from {VIDEO_PATH}...')
frames_bgr, fps = extract_frames(VIDEO_PATH)
print(f'  {len(frames_bgr)} frames @ {fps:.1f} fps')

# Detect product bbox from first frame, reuse for all frames
print('Detecting product...')
cached_bbox = get_product_bbox(frames_bgr[0], point=SAM_POINT)
x1,y1,x2,y2 = cached_bbox
print(f'  Product bbox: ({x1},{y1}) → ({x2},{y2})')

output_frames = []

for i, frame_bgr in enumerate(tqdm(frames_bgr, desc='Generating hands')):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    h, w = frame_rgb.shape[:2]

    # Re-detect bbox every 30 frames for moving products
    if i % 30 == 0 and i > 0:
        try:
            cached_bbox = get_product_bbox(frame_bgr, point=SAM_POINT)
        except Exception:
            pass

    pose_img, mask, _, _ = generate_pose_and_mask(
        (h, w), cached_bbox, hand_scale=HAND_SIZE, curl=FINGER_CURL
    )

    result_rgb = generate_hands(
        frame_rgb=frame_rgb,
        mask=mask,
        pose_img=pose_img,
        skin_tone=SKIN_TONE,
        steps=INFERENCE_STEPS,
        seed=SEED + i,
    )

    output_frames.append(cv2.cvtColor(result_rgb, cv2.COLOR_RGB2BGR))

print('Saving video...')
save_video(output_frames, OUTPUT_PATH, fps)
print(f'\n✓ Done! Output saved to {OUTPUT_PATH}')

In [ ]:
# ── Cell 9: Preview output ────────────────────────────────────────────────────
from IPython.display import HTML
import base64

with open(OUTPUT_PATH, 'rb') as f:
    video_b64 = base64.b64encode(f.read()).decode()

HTML(f'''
<video width="720" controls>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
''')

In [ ]:
# ── Cell 10: Download output video ────────────────────────────────────────────
from google.colab import files
files.download(OUTPUT_PATH)
print('✓ Download started!')

---
## Tips & Troubleshooting

| Problem | Fix |
|---|---|
| Hands in wrong position | Set `SAM_POINT = (x, y)` in Cell 6 where the product center is |
| Hands too small/large | Increase/decrease `HAND_SIZE` (try 0.12–0.25) |
| Fingers look flat | Increase `FINGER_CURL` to 0.45–0.55 |
| Low quality hands | Increase `INFERENCE_STEPS` to 35–40 |
| Out of memory | Restart runtime, re-run from Cell 3 |
| Very long video | Trim to under 10 sec first using: `ffmpeg -ss 0 -t 10 -i input.mp4 trimmed.mp4` |
| Session disconnects | Mount Google Drive in Cell 5 instead of uploading directly |

---
*Processing time on T4: ~20 sec/frame. A 5-second video at 30fps = ~50 min.*  
*For speed: use 24fps source video, or set `INFERENCE_STEPS = 20`.*